In [ ]:
import yaml
import json
import os

from src.datasets.dataset_from_tg import data_full_routine
from src.transformers.transformers_utils import init_pretrained_model, tokenize_function

In [2]:
CONFIG_DIR = "configs/tune"
CONFIG_NAME = "rut5_small.test.yaml" # parse arg
CONFIG_PATH = os.path.join(CONFIG_DIR, CONFIG_NAME)

In [3]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
    
random_seed = config["RANDOM_SEED"]
model_name = config["MODEL_NAME"]

data_path = os.path.join(config["DATA_DIR"], config["DATA_NAME"])
with open(data_path, "r") as f:
    raw_data = json.load(f)

name = config["RESPONSE_NAME"]

In [ ]:
# model, tokenizer = init_pretrained_model(model_name, random_seed, device_map="cuda:0")
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("cointegrated/rut5-small-chitchat")
model = T5ForConditionalGeneration.from_pretrained("cointegrated/rut5-small-chitchat")

dataset = data_full_routine(raw_data, name)
# tokenized_dataset = dataset.map(lambda x: tokenize_function(sample=x, tokenizer=tokenizer), batched=True)

In [23]:
from datasets import Dataset
dataset = Dataset.from_dict(dataset[:10000])

In [24]:
train_test_split = dataset.train_test_split(test_size=0.3, shuffle=True, )
data_train = train_test_split["train"]
data_eval = train_test_split["test"]

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['context'], padding="max_length", truncation=True)

def preprocess_function(examples):
    inputs = [f"dialogue: {context} </s>" for context in examples["context"]]
    targets = [f"{response} </s>" for response in examples["response"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Настройка labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

data_train = data_train.map(preprocess_function, batched=True)
data_eval = data_eval.map(preprocess_function, batched=True)

In [ ]:
data_train = data_train.map(tokenize_function, batched=True)
data_eval = data_eval.map(tokenize_function, batched=True)

In [ ]:
model.eval()

In [28]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
data_eval["response"]

In [ ]:
data_eval

In [ ]:
model

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./temp",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data_train,
    eval_dataset=data_eval,  # Можно добавить валидационный датасет
    data_collator=data_collator,
)

trainer.train()

In [ ]:
SAVE_DIR = "artifacts/20241208_rut5_test"

model.save_pretrained(SAVE_DIR + "/model")
tokenizer.save_pretrained(SAVE_DIR + "/tokenizer")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('path/to/save/model')
tokenizer = T5Tokenizer.from_pretrained('path/to/save/model')